In [1]:
# from core.XGBoostModel import XGBoostModel
# from core.RunData import RunPipeline
import pandas as pd
import optuna
from sklearn.preprocessing import LabelEncoder
import numpy as np
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold ,train_test_split, cross_val_score
import kagglehub
import os
import zipfile
# from core.seed_utils import SEED
import requests


/home/daniel7/.conda/envs/my_env_311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# GenericDataPipeline

In [ ]:
class GenericDataPipeline:

    def rank_features(self, X, y, n_folds=5):
        print(f"Starting feature importance calculation with XGBoost and {n_folds}-fold CV...")

        X_copy = X.copy()

        null_ratios = {}
        for col in X_copy.columns:
            null_ratios[col] = X_copy[col].isna().mean()

        for col in X_copy.select_dtypes(include=['int64', 'float64']).columns:
            X_copy[col] = X_copy[col].replace([np.inf, -np.inf], np.nan)
            
            non_null = X_copy[col].dropna()
            if len(non_null) > 0:
                mean_val = non_null.mean()
                std_val = non_null.std()
                if std_val > 0 and not np.isnan(mean_val):
                    upper_bound = mean_val + 5 * std_val
                    lower_bound = mean_val - 5 * std_val
                    X_copy[col] = X_copy[col].clip(lower=lower_bound, upper=upper_bound)
        
        num_cols = X_copy.select_dtypes(include=['int64', 'float64']).columns
        cat_cols = X_copy.select_dtypes(include=['object']).columns
        
        X_processed = X_copy.copy()
        
        for col in cat_cols:
            if X_processed[col].isna().sum() > 0:
                most_freq = X_processed[col].mode()[0] if not X_processed[col].isna().all() else 'Unknown'
                X_processed[col] = X_processed[col].fillna(most_freq)
            X_processed[col] = X_processed[col].astype('category')
        
        for col in num_cols:
            if X_processed[col].isna().sum() > 0:
                median_val = X_processed[col].median() if not X_processed[col].isna().all() else 0
                X_processed[col] = X_processed[col].fillna(median_val)
        
        X_filled = X_processed

        mean_target = float(np.mean(y))
        print(f"Mean target value (for base_score): {mean_target:.6f}")

        base_score = 0.5
        if 0 < mean_target < 1:
            base_score = mean_target

        params = {
            'objective': 'binary:logistic',
            'eval_metric': 'auc',
            'max_depth': 6,
            'eta': 0.1,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'min_child_weight': 3,
            'alpha': 0.01,
            'lambda': 1.0,
            'gamma': 0.1,
            'base_score': base_score, 
            'seed': 42,
            'tree_method': 'hist',
            'device': 'cuda'

        }

        kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

        fold_importances = []

        fold_scores = []

        print(f"Training XGBoost models across {n_folds} folds...")

        y_values = set(y.unique())
        print(f"Unique target values: {y_values}")

        if not all(isinstance(val, (int, float)) for val in y_values):
            print("Warning: Target variable contains non-numeric values. Converting to numeric.")
            y = y.astype(float)

        if not all(val in [0, 1] for val in y_values):
            print("Warning: Target variable contains values other than 0 and 1.")
            print("Converting target to binary: 0 for negative class, 1 for positive class.")
            y = (y > 0).astype(int)

        # Perform cross-validation
        for fold, (train_idx, val_idx) in enumerate(kf.split(X_filled, y)):
            print(f"Fold {fold+1}/{n_folds}")

            X_train, X_val = X_filled.iloc[train_idx], X_filled.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

            train_pos = np.mean(y_train)
            val_pos = np.mean(y_val)
            print(f"  Train positive rate: {train_pos:.4f}, Validation positive rate: {val_pos:.4f}")

            try:
                dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
                dval = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)

                # Train model
                model = xgb.train(
                    params,
                    dtrain,
                    num_boost_round=100,
                    early_stopping_rounds=20,
                    evals=[(dval, 'validation')],
                    verbose_eval=False
                )

                y_pred = model.predict(dval)
                score = roc_auc_score(y_val, y_pred)
                fold_scores.append(score)

                try:
                    importance = model.get_score(importance_type='gain')

                    fold_importance = {col: importance.get(col, 0) for col in X_filled.columns}

                    max_value = max(fold_importance.values()) if max(fold_importance.values()) > 0 else 1
                    normalized_importance = {col: val/max_value for col, val in fold_importance.items()}

                    fold_importances.append(normalized_importance)
                except Exception as e:
                    print(f"Warning: Could not get feature importance for fold {fold+1}: {str(e)}")

            except Exception as e:
                print(f"Error in fold {fold+1}: {str(e)}")
                print("Skipping this fold and continuing with others.")

        if fold_scores:
            mean_auc = np.mean(fold_scores)
            print(f"Mean AUC across {len(fold_scores)} successful folds: {mean_auc:.4f}")
        else:
            print("No successful folds. Cannot calculate mean AUC.")

        if not fold_importances:
            print("No feature importance information could be gathered from any fold.")
            print("Using a simple fallback method for feature scoring.")

            avg_importance = {}
            for col in X_filled.columns:
                try:
                    valid_mask = ~pd.isna(X_copy[col]) & ~pd.isna(y)
                    if valid_mask.sum() > 10: 
                        if X_filled[col].dtype.name == 'category':
                            col_values = X_filled[col][valid_mask].cat.codes
                        else:
                            col_values = X_filled[col][valid_mask]
                        
                        y_values = y[valid_mask]
                        corr = abs(np.corrcoef(col_values, y_values)[0, 1])
                        avg_importance[col] = corr if not np.isnan(corr) else 0
                    else:
                        avg_importance[col] = np.random.uniform(0, 0.001)
                except Exception as e:
                    print(f"Warning: Could not calculate correlation for {col}: {str(e)}")
                    avg_importance[col] = np.random.uniform(0, 0.001)
        else:
            avg_importance = {}
            for col in X_filled.columns:
                importances = [fold_imp.get(col, 0) for fold_imp in fold_importances]
                if importances:
                    avg_importance[col] = np.mean(importances)
                else:
                    null_ratio = null_ratios.get(col, 0)
                    avg_importance[col] = max(0.001 * (1 - null_ratio), 0.0001)  

        feature_data = []
        for col in X_copy.columns:
            gain_score = avg_importance.get(col, 0)
            null_ratio = null_ratios.get(col, 0)
            feature_data.append({
                'feature_name': col,
                'gain_score': gain_score,
                'null_ratio': null_ratio
            })
        
        feature_df = pd.DataFrame(feature_data)
        feature_df = feature_df.sort_values('gain_score', ascending=False)
        
        print("\nFeature scores:")
        print("-" * 100)
        print(f"{'Rank':<5} | {'Feature':<30} | {'Gain Score':<15} | {'Null Ratio':<10}")
        print("-" * 100)

        for i, row in feature_df.iterrows():
            rank = i + 1
            feat = row['feature_name']
            gain = row['gain_score']
            null_ratio = row['null_ratio']
            print(f"{rank:<5} | {feat:<30} | {gain:.6f} | {null_ratio:.4f}")

        return feature_df

    def objective(self, trial, feature_scores_with_nulls, all_features_score, df, label="readmitted"):
        K = 10
        selected_features = feature_scores_with_nulls['feature_name'].to_list()[:K]
        print("selected_features")
        print(selected_features)
        all_features = all_features_score['feature_name'].to_list()
        # Create binary assignment for each of the selected features:
        assignment = {}
        for feat in selected_features:
            # 1 means feature goes to stage1, 0 means feature is used in stage2
            assignment[feat] = trial.suggest_categorical(f"assign_{feat}", [1, 0])
        
        for feat in all_features:
            if feat not in assignment.keys():
                assignment[feat] = 1
        vals = 0
        for key,value in assignment.items():
            vals +=value
        if vals==0:
            return 99999
        penalty = 0 
        # Derive feature sets based on the assignment:
        stage1_features = [feat for feat, assign in assignment.items() if assign == 1]
        stage2_features = [feat for feat, assign in assignment.items() if assign == 0]
        csv_name = f"trial_{trial.number}_results.csv"
        dm = RunPipeline()
        validation_loss = dm.full_run(df,stage1_features,stage2_features,label,csv_name)
        if validation_loss == 999:
            return validation_loss
        else:
            final_objective = validation_loss + penalty
            return final_objective
    def preprocessing(self,df):
        for col in df.columns:
            if df[col].dtype == 'object':
                # Replace ? with NaN
                df[col] = df[col].replace(['?', ''], np.nan)
                # Convert to categorical codes
                if df[col].isna().sum() < len(df):  # If not all values are NaN
                    df[col] = pd.Categorical(df[col]).codes

                    # Ensure -1 (missing) values are converted to NaN
                    df[col] = df[col].replace(-1, np.nan)
        # Convert boolean columns to int
        for col in df.select_dtypes(include=['bool']).columns:
            df[col] = df[col].astype(int)
        return df 

    def movie_land(self,n_trials=20):
        ratings = pd.read_csv('ml-25m/ml-25m/ratings.csv')
        movies = pd.read_csv('ml-25m/ml-25m/movies.csv')
        tags = pd.read_csv('ml-25m/ml-25m/tags.csv')

        cutoff_date = ratings['timestamp'].quantile(0.8)
        target_date = ratings['timestamp'].quantile(0.9)
    
        # Base rating features
        user_stats = ratings[ratings['timestamp'] < cutoff_date].groupby('userId').agg({
            'rating': ['count', 'mean', 'std'],
            'timestamp': ['min', 'max']
        }).round(3)
    
        user_stats.columns = ['rating_count', 'rating_mean', 'rating_std', 
                              'first_rating', 'last_rating']
        df = user_stats.reset_index()
        df['days_active'] = (df['last_rating'] - df['first_rating']) / (24 * 60 * 60)
        df['rating_frequency'] = df['rating_count'] / df['days_active'].clip(lower=1)
    
        future_activity = ratings[
            (ratings['timestamp'] >= cutoff_date) & 
            (ratings['timestamp'] < target_date)
        ].groupby('userId')['rating'].count().reset_index()
        future_activity.columns = ['userId', 'future_ratings']
        future_activity['TARGET'] = (future_activity['future_ratings'] > 
                                     future_activity['future_ratings'].median()).astype(int)
        
        df = df.merge(future_activity[['userId', 'TARGET']], on='userId', how='left')
        df['TARGET'] = df['TARGET'].fillna(0)
    
        tag_activity = tags[tags['timestamp'] < cutoff_date].groupby('userId').agg({
            'tag': ['count', 'nunique'],
            'timestamp': ['min', 'max']
        })
        tag_activity.columns = ['tag_count', 'unique_tags', 'first_tag', 'last_tag']
        tag_activity = tag_activity.reset_index()
    
        # Add tag frequency
        tag_activity['days_tagging'] = (tag_activity['last_tag'] - tag_activity['first_tag']) / (24 * 60 * 60)
        tag_activity['tag_frequency'] = tag_activity['tag_count'] / tag_activity['days_tagging'].clip(lower=1)
    
        # Add avg tag length
        tag_lengths = tags[tags['timestamp'] < cutoff_date].groupby('userId')['tag'].apply(
            lambda x: np.mean([len(str(t)) for t in x])
        ).reset_index()
        tag_lengths.columns = ['userId', 'avg_tag_length']
    
        tag_activity = tag_activity.merge(tag_lengths, on='userId', how='left')
    
        df = df.merge(tag_activity, on='userId', how='left')
        columns_to_keep = [
            'rating_count', 'rating_mean', 'rating_std',
            'days_active', 'rating_frequency',
            'tag_count', 'unique_tags', 'avg_tag_length',
            'tag_frequency', 'last_tag','TARGET'
        ]
        
        df = df[columns_to_keep]

        df = self.preprocessing(df)
        return df

# movie_land

In [5]:

fs = GenericDataPipeline()
study = fs.movie_land(20)

Using 'Customer Status' as target variable
Starting feature importance calculation with XGBoost and 5-fold CV...
Mean target value (for base_score): 0.012344
Training XGBoost models across 5 folds...
Unique target values: {np.float64(0.0), np.float64(1.0)}
Fold 1/5
  Train positive rate: 0.0123, Validation positive rate: 0.0123
Fold 2/5
  Train positive rate: 0.0123, Validation positive rate: 0.0124
Fold 3/5
  Train positive rate: 0.0123, Validation positive rate: 0.0124
Fold 4/5
  Train positive rate: 0.0123, Validation positive rate: 0.0123
Fold 5/5
  Train positive rate: 0.0123, Validation positive rate: 0.0123


[I 2025-05-18 12:16:02,182] A new study created in memory with name: no-name-a0321eb1-a605-40b1-980f-303f7f5105da


Mean AUC across 5 successful folds: 0.9664

Feature scores:
----------------------------------------------------------------------------------------------------
Rank  | Feature                        | Gain Score      | Null Ratio
----------------------------------------------------------------------------------------------------
6     | last_rating                    | 1.000000 | 0.0000
2     | rating_count                   | 0.311861 | 0.0000
7     | days_active                    | 0.182639 | 0.0000
8     | rating_frequency               | 0.137395 | 0.0000
5     | first_rating                   | 0.110085 | 0.0000
13    | days_tagging                   | 0.057049 | 0.9168
12    | last_tag                       | 0.054643 | 0.9168
9     | tag_count                      | 0.052359 | 0.9168
15    | avg_tag_length                 | 0.050970 | 0.9168
10    | unique_tags                    | 0.049965 | 0.9168
3     | rating_mean                    | 0.046282 | 0.0000
14    | tag_frequen

[I 2025-05-18 12:16:02,576] A new study created in memory with name: no-name-ef089d0d-47ff-42b9-8fef-0d5729798afd


Train set distribution:
TARGET  has_extended
0.0     0               100415
        1                 8530
1.0     0                  719
        1                  642
dtype: int64

Test set distribution:
TARGET  has_extended
0.0     0               25104
        1                2132
1.0     0                 180
        1                 161
dtype: int64

=== Training Base Model ===


...

********** run results **********
{'max_depth': 5, 'learning_rate': 0.007353902158168152, 'n_estimators': 741, 'min_child_weight': 3, 'subsample': 0.7067985540237419, 'colsample_bytree': 0.8537967251193104, 'gamma': 2.158034732558517, 'lambda': 5.906195831814526, 'alpha': 5.668476476388617, 'scale_pos_weight': 1.0571508518933976, 'random_state': 42, 'eval_metric': 'auc', 'tree_method': 'hist', 'device': 'cuda'}
*****
[np.float64(0.9602033534245993), np.float64(0.9689918599460112), np.float64(0.9708624416849138)]
********** run results **********


/sise/home/daniel7/IncrementalLearing/Changes /XGBoostModel.py:21: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.1),
[I 2025-05-18 13:49:35,854] Trial 19 finished with value: 0.9593891519443577 and parameters: {'max_depth': 3, 'learning_rate': 0.0010636747773539543, 'n_estimators': 573, 'min_child_weight': 3, 'subsample': 0.9960350042319281, 'colsample_bytree': 0.9374887483680339, 'gamma': 1.0834011360669733, 'lambda': 1.177172240995346, 'alpha': 9.989834817914986, 'scale_pos_weight': 2.906503858276733}. Best is trial 11 with value: 0.8111459890742179.


********** run results **********
{'max_depth': 3, 'learning_rate': 0.0010636747773539543, 'n_estimators': 573, 'min_child_weight': 3, 'subsample': 0.9960350042319281, 'colsample_bytree': 0.9374887483680339, 'gamma': 1.0834011360669733, 'lambda': 1.177172240995346, 'alpha': 9.989834817914986, 'scale_pos_weight': 2.906503858276733, 'random_state': 42, 'eval_metric': 'auc', 'tree_method': 'hist', 'device': 'cuda'}
*****
[np.float64(0.9525811796004428), np.float64(0.96113324298522), np.float64(0.9644530332474106)]
********** run results **********
Best parameters found: {'max_depth': 3, 'learning_rate': 0.0010836297723846497, 'n_estimators': 982, 'min_child_weight': 5, 'subsample': 0.9444251483703959, 'colsample_bytree': 0.9535555626695151, 'gamma': 1.484275296479742, 'lambda': 3.2863613362573236, 'alpha': 6.9199191028461655, 'scale_pos_weight': 0.14138475577687082}
Best CV AUC score: 0.8111

===== Detailed Cross-Validation Results with Best Parameters =====
Params: {'max_depth': 3, 'lear

[I 2025-05-18 13:51:13,658] Trial 19 finished with value: -0.18892770732822817 and parameters: {'assign_days_tagging': 0, 'assign_last_tag': 0, 'assign_tag_count': 1, 'assign_avg_tag_length': 0, 'assign_unique_tags': 0, 'assign_tag_frequency': 0, 'assign_first_tag': 0}. Best is trial 9 with value: -0.34212054906735045.



Extended model (with extended)
AUC: 0.9201, Accuracy: 0.9276, F1 Score: 0.5787

Combined model (no extended)
AUC: 0.8158, Accuracy: 0.9929, F1 Score: 0.0000

Combined model (with extended)
AUC: 0.8733, Accuracy: 0.9298, F1 Score: 0.0000

Results Summary:
                            Model       AUC  Accuracy        F1  \
0        Base model (no extended)  0.958000  0.991853  0.386905   
1  Extended model (with extended)  0.920091  0.927606  0.578680   
2    Combined model (no extended)  0.815833  0.992881  0.000000   
3  Combined model (with extended)  0.873329  0.929786  0.000000   

                                                                                                        Base_Features  \
0  tag_count, last_rating, rating_count, days_active, rating_frequency, first_rating, rating_mean, rating_std, userId   
1  tag_count, last_rating, rating_count, days_active, rating_frequency, first_rating, rating_mean, rating_std, userId   
2  tag_count, last_rating, rating_count, days

In [2]:
# Define the input data
data = {
    "Model": [
        "Base model (no extended)",
        "Extended model (with extended)",
        "Combined model (no extended)",
        "Combined model (with extended)",
    ],
    "AUC": [0.940120, 0.919033, 0.720208, 0.796824],
    "Base_Features": [
        "tag_count, unique_tags, last_rating, rating_count, days_active, rating_frequency, first_rating, rating_mean, rating_std, userId"
    ] * 4,
    "Extended_Features": [
        "days_tagging, last_tag, avg_tag_length, tag_frequency, first_tag, has_extended"
    ] * 4,
}

df = pd.DataFrame(data)

# Display AUC summary
print("AUC Summary:")
print(df[["Model", "AUC"]].to_string(index=False))

# Show one example of features
print("\nBase Features:")
print(df.loc[0, "Base_Features"])

print("\nExtended Features:")
print(df.loc[0, "Extended_Features"])

# Calculate AUC differences
diff_ext_comb = df.loc[1, "AUC"] - df.loc[3, "AUC"]
diff_base_comb = df.loc[0, "AUC"] - df.loc[2, "AUC"]

# Standard deviations
std_ext = 0.0042
std_base = 0.0064

# Print results
print("\nAUC Differences:")
print(f"Extended model - Combined (with extended): {diff_ext_comb}")
print(f"Base model - Combined (no extended):       {diff_base_comb}")

print("\nStandard Deviations:")
print(f"Extended model - Std Dev: {std_ext}")
print(f"Base model - Std Dev:     {std_base}")


AUC Summary:
                         Model      AUC
      Base model (no extended) 0.940120
Extended model (with extended) 0.919033
  Combined model (no extended) 0.720208
Combined model (with extended) 0.796824

Base Features:
tag_count, unique_tags, last_rating, rating_count, days_active, rating_frequency, first_rating, rating_mean, rating_std, userId

Extended Features:
days_tagging, last_tag, avg_tag_length, tag_frequency, first_tag, has_extended

AUC Differences:
Extended model - Combined (with extended): 0.12220900000000001
Base model - Combined (no extended):       0.219912

Standard Deviations:
Extended model - Std Dev: 0.0042
Base model - Std Dev:     0.0064
